The script below builds one 10 m master grid from the 1 km buffer in EPSG:32645, which covers the whole flood area and stays within UTM zone 45N. It tiles that grid into 10 km boxes, picks the least cloudy date per box and period using in-box cloud cover from the SCL band, and downloads only the clipped B02, B03, and B04 windows into s2_all\{t1,t2}\scene###\. Boxes with a cloud cover above 10% trigger a warning, and the old s2_tg folders are never touched.

In [ ]:
import math
import warnings
from pathlib import Path

import geopandas as gpd
import numpy as np
import planetary_computer as pc
import rasterio
from pystac_client import Client
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.transform import Affine
from rasterio.warp import transform_bounds
from rasterio.windows import Window, from_bounds
from rasterio.windows import transform as window_transform
from shapely.geometry import box as shapely_box
from shapely.ops import unary_union

# ---------------------------------------------------------------------------
# Paths (relative to the project folder you run the script from)
# ---------------------------------------------------------------------------

PROJECT_DIR = Path.cwd()
PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
VECTOR_PATH = RAW_DATA_DIR / "vectors" / "nepal_flood_extent_2026-08-27.geojson"
S2_ROOT = RAW_DATA_DIR / "s2_all"

# ---------------------------------------------------------------------------
# Settings
# ---------------------------------------------------------------------------
UTM_EPSG = 32645
CRS = f"EPSG:{UTM_EPSG}"
PIXEL_SIZE = 10            # metres, master grid and Sentinel-2 B02/B03/B04
BUFFER_M = 1000            # metres, applied after reprojecting to UTM
BOX_SIZE_M = 10_240        # metres, tile size
STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
COLLECTION = "sentinel-2-l2a"
BANDS = ["B02", "B03", "B04"]
SCL_CLOUD_CLASSES = [8, 9, 10]   # medium cloud, high cloud, cirrus
CLOUD_WARN_PCT = 10.0
PERIODS = {
    "t1": ("2026-07-11", "2026-08-25"),   # 1.5 months before the 26 Aug 2026 flood
    "t2": ("2026-08-26", "2026-10-11"),   # 1.5 months after the flood
}


# ---------------------------------------------------------------------------
# Grid helpers
# ---------------------------------------------------------------------------
def make_master_grid(bounds):
    """10 m grid whose origin is snapped to multiples of 10 m."""
    minx, miny, maxx, maxy = bounds
    x0 = math.floor(minx / PIXEL_SIZE) * PIXEL_SIZE
    y_top = math.ceil(maxy / PIXEL_SIZE) * PIXEL_SIZE
    y_bottom = math.floor(miny / PIXEL_SIZE) * PIXEL_SIZE
    width = math.ceil((maxx - x0) / PIXEL_SIZE)
    height = math.ceil((y_top - y_bottom) / PIXEL_SIZE)
    transform = Affine(PIXEL_SIZE, 0, x0, 0, -PIXEL_SIZE, y_top)
    return transform, width, height


def make_boxes(buffer_geom, transform, width, height):
    """10 km boxes on a regular UTM grid, clipped to the master grid extent."""
    left = transform.c
    top = transform.f
    right = left + width * PIXEL_SIZE
    bottom = top - height * PIXEL_SIZE

    minx, miny, maxx, maxy = buffer_geom.bounds
    x_starts = np.arange(math.floor(minx / BOX_SIZE_M) * BOX_SIZE_M, maxx, BOX_SIZE_M)
    y_starts = np.arange(math.floor(miny / BOX_SIZE_M) * BOX_SIZE_M, maxy, BOX_SIZE_M)

    boxes = []
    for y in y_starts:
        for x in x_starts:
            cell = shapely_box(x, y, x + BOX_SIZE_M, y + BOX_SIZE_M)
            if not cell.intersects(buffer_geom):
                continue
            boxes.append((
                max(x, left),
                max(y, bottom),
                min(x + BOX_SIZE_M, right),
                min(y + BOX_SIZE_M, top),
            ))
    return boxes


def box_window(bounds, transform):
    """Pixel window of a box on the master grid, with integer offsets."""
    win = from_bounds(*bounds, transform=transform)
    return Window(round(win.col_off), round(win.row_off),
                  round(win.width), round(win.height))


# ---------------------------------------------------------------------------
# Raster reading and mosaicking
# ---------------------------------------------------------------------------
def read_window(href, bounds, out_h, out_w, resampling):
    """Read only the box window from one Sentinel-2 COG."""
    with rasterio.open(href) as src:
        if src.crs is None or src.crs.to_epsg() != UTM_EPSG:
            return None  # tile from another UTM zone, skip
        win = from_bounds(*bounds, transform=src.transform)
        win = Window(round(win.col_off), round(win.row_off),
                     round(win.width), round(win.height))
        return src.read(
            1,
            window=win,
            out_shape=(out_h, out_w),
            boundless=True,
            fill_value=0,
            resampling=resampling,
        )


def mosaic(items, asset_key, bounds, out_h, out_w, resampling, dtype):
    """Combine the box windows of several same-day tiles; 0 = no data."""
    result = np.zeros((out_h, out_w), dtype=dtype)
    for item in items:
        if asset_key not in item.assets:
            continue
        arr = read_window(item.assets[asset_key].href, bounds, out_h, out_w, resampling)
        if arr is None:
            continue
        fill = (result == 0) & (arr != 0)
        result[fill] = arr[fill]
    return result


def cloud_fraction(day_items, bounds, out_h, out_w):
    """In-box cloud cover (%) from the SCL band, ignoring no-data pixels."""
    scl = mosaic(day_items, "SCL", bounds, out_h, out_w, Resampling.nearest, np.uint8)
    valid = scl != 0
    n_valid = int(valid.sum())
    if n_valid == 0:
        return None
    cloud = np.isin(scl, SCL_CLOUD_CLASSES) & valid
    return 100.0 * int(cloud.sum()) / n_valid


def group_by_date(items):
    groups = {}
    for item in items:
        groups.setdefault(item.datetime.date().isoformat(), []).append(item)
    return groups


def best_date(items, bounds, out_h, out_w):
    """Return (date, cloud_pct) for the least cloudy date inside the box."""
    best = None
    for day, day_items in group_by_date(items).items():
        pct = cloud_fraction(day_items, bounds, out_h, out_w)
        if pct is None:
            continue
        if best is None or pct < best[1]:
            best = (day, pct)
    return best


def write_band(path, data, transform):
    with rasterio.open(
        path, "w",
        driver="GTiff",
        height=data.shape[0],
        width=data.shape[1],
        count=1,
        dtype=data.dtype,
        crs=CRS,
        transform=transform,
        nodata=0,
    ) as dst:
        dst.write(data, 1)


# ---------------------------------------------------------------------------
# Main workflow
# ---------------------------------------------------------------------------
def main():
    gdf = gpd.read_file(VECTOR_PATH)
    if gdf.crs is None:
        raise ValueError("The flood vector has no CRS defined.")
    gdf = gdf.to_crs(UTM_EPSG)

    # Buffer after reprojecting to UTM so the 1 km is in metres
    buffer_geom = unary_union(gdf.geometry.buffer(BUFFER_M))

    # One master grid for the flood mask and all boxes
    transform, width, height = make_master_grid(buffer_geom.bounds)

    S2_ROOT.mkdir(parents=True, exist_ok=True)

    # Rasterize the (unbuffered) flood extent once on the master grid
    flood_shapes = [
        (geom, 1) for geom in gdf.geometry
        if geom is not None and not geom.is_empty
    ]
    flood = rasterize(
        flood_shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0,
        all_touched=False,
        dtype="uint8",
    )
    mask_path = S2_ROOT / "flood_mask_master.tif"
    with rasterio.open(
        mask_path, "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="uint8",
        crs=CRS,
        transform=transform,
        nodata=0,
    ) as dst:
        dst.write(flood, 1)
    print(f"Master flood mask written: {mask_path}")

    boxes = make_boxes(buffer_geom, transform, width, height)
    print(f"{len(boxes)} boxes of {BOX_SIZE_M // 1000} km cover the buffered flood area.")

    catalog = Client.open(STAC_URL, modifier=pc.sign_inplace)

    summary = []
    for idx, bounds in enumerate(boxes, start=1):
        win = box_window(bounds, transform)
        out_h, out_w = int(win.height), int(win.width)
        out_transform = window_transform(win, transform)
        bbox_ll = transform_bounds(CRS, "EPSG:4326", *bounds)

        for period, (start, end) in PERIODS.items():
            out_dir = S2_ROOT / period / f"scene{idx:03d}"
            label = f"scene{idx:03d} {period}"

            if all((out_dir / f"{band}.tif").exists() for band in BANDS):
                print(f"{label}: already downloaded, skipping.")
                continue

            search = catalog.search(
                collections=[COLLECTION],
                bbox=bbox_ll,
                datetime=f"{start}/{end}",
            )
            items = list(search.items())
            if not items:
                msg = f"{label}: no Sentinel-2 items found in {start} to {end}."
                warnings.warn(msg, UserWarning)
                summary.append((label, None, None, "no items"))
                continue

            result = best_date(items, bounds, out_h, out_w)
            if result is None:
                msg = f"{label}: no usable pixels inside the box."
                warnings.warn(msg, UserWarning)
                summary.append((label, None, None, "no usable pixels"))
                continue

            best_day, pct = result
            day_items = group_by_date(items)[best_day]

            out_dir.mkdir(parents=True, exist_ok=True)
            for band in BANDS:
                data = mosaic(
                    day_items, band, bounds, out_h, out_w,
                    Resampling.nearest, np.uint16,
                )
                write_band(out_dir / f"{band}.tif", data, out_transform)

            print(f"{label}: date {best_day}, in-box cloud cover {pct:.1f}%")
            if pct > CLOUD_WARN_PCT:
                msg = (f"{label}: in-box cloud cover {pct:.1f}% exceeds "
                       f"{CLOUD_WARN_PCT:.0f}% threshold (date {best_day}).")
                warnings.warn(msg, UserWarning)
            summary.append((label, best_day, pct, "downloaded"))

    print("\nSummary")
    print(f"{'Box':<22}{'Date':<14}{'Cloud %':>10}  Status")
    for label, day, pct, status in summary:
        pct_str = f"{pct:.1f}" if pct is not None else "-"
        print(f"{label:<22}{str(day or '-'):<14}{pct_str:>10}  {status}")


if __name__ == "__main__":
    main()


Master flood mask written: c:\Users\herol\Documents\sds320_project\data\raw\s2_all\flood_mask_master.tif
21 boxes of 10 km cover the buffered flood area.
scene001 t1: already downloaded, skipping.
scene001 t2: already downloaded, skipping.
scene002 t1: already downloaded, skipping.
scene002 t2: already downloaded, skipping.
scene003 t1: already downloaded, skipping.
scene003 t2: already downloaded, skipping.
scene004 t1: already downloaded, skipping.
scene004 t2: already downloaded, skipping.
scene005 t1: already downloaded, skipping.
scene005 t2: already downloaded, skipping.
scene006 t1: already downloaded, skipping.
scene006 t2: already downloaded, skipping.
scene007 t1: already downloaded, skipping.
scene007 t2: already downloaded, skipping.
scene008 t1: already downloaded, skipping.
scene008 t2: date 2026-10-23, in-box cloud cover 0.0%
scene009 t1: date 2026-08-12, in-box cloud cover 10.2%


C:\Users\herol\AppData\Local\Temp\ipykernel_34424\2900022119.py:277: UserWarning: scene009 t1: in-box cloud cover 10.2% exceeds 10% threshold (date 2026-08-12).
  warnings.warn(msg, UserWarning)


scene009 t2: date 2026-10-26, in-box cloud cover 0.0%
scene010 t1: date 2026-08-12, in-box cloud cover 11.9%


C:\Users\herol\AppData\Local\Temp\ipykernel_34424\2900022119.py:277: UserWarning: scene010 t1: in-box cloud cover 11.9% exceeds 10% threshold (date 2026-08-12).
  warnings.warn(msg, UserWarning)


scene010 t2: date 2026-10-26, in-box cloud cover 0.0%
scene011 t1: date 2026-08-12, in-box cloud cover 3.4%
scene011 t2: date 2026-10-26, in-box cloud cover 0.0%
scene012 t1: date 2026-08-12, in-box cloud cover 16.2%


C:\Users\herol\AppData\Local\Temp\ipykernel_34424\2900022119.py:277: UserWarning: scene012 t1: in-box cloud cover 16.2% exceeds 10% threshold (date 2026-08-12).
  warnings.warn(msg, UserWarning)


scene012 t2: date 2026-10-26, in-box cloud cover 0.0%
scene013 t1: date 2026-08-12, in-box cloud cover 0.9%
scene013 t2: date 2026-10-26, in-box cloud cover 0.0%
scene014 t1: date 2026-08-12, in-box cloud cover 4.2%
scene014 t2: date 2026-10-26, in-box cloud cover 0.0%
scene015 t1: date 2026-08-12, in-box cloud cover 1.3%
scene015 t2: date 2026-10-26, in-box cloud cover 0.3%
scene016 t1: date 2026-08-12, in-box cloud cover 2.1%
scene016 t2: date 2026-10-26, in-box cloud cover 0.0%
scene017 t1: date 2026-08-12, in-box cloud cover 1.3%
scene017 t2: date 2026-10-26, in-box cloud cover 0.0%
scene018 t1: date 2026-08-12, in-box cloud cover 2.8%
scene018 t2: date 2026-10-26, in-box cloud cover 0.0%
scene019 t1: date 2026-08-12, in-box cloud cover 0.0%
scene019 t2: date 2026-10-26, in-box cloud cover 0.0%
scene020 t1: date 2026-08-12, in-box cloud cover 26.2%


C:\Users\herol\AppData\Local\Temp\ipykernel_34424\2900022119.py:277: UserWarning: scene020 t1: in-box cloud cover 26.2% exceeds 10% threshold (date 2026-08-12).
  warnings.warn(msg, UserWarning)


scene020 t2: date 2026-10-26, in-box cloud cover 0.0%
scene021 t1: date 2026-08-12, in-box cloud cover 0.0%
scene021 t2: date 2026-10-26, in-box cloud cover 0.0%

Summary
Box                   Date             Cloud %  Status
scene008 t2           2026-10-23           0.0  downloaded
scene009 t1           2026-08-12          10.2  downloaded
scene009 t2           2026-10-26           0.0  downloaded
scene010 t1           2026-08-12          11.9  downloaded
scene010 t2           2026-10-26           0.0  downloaded
scene011 t1           2026-08-12           3.4  downloaded
scene011 t2           2026-10-26           0.0  downloaded
scene012 t1           2026-08-12          16.2  downloaded
scene012 t2           2026-10-26           0.0  downloaded
scene013 t1           2026-08-12           0.9  downloaded
scene013 t2           2026-10-26           0.0  downloaded
scene014 t1           2026-08-12           4.2  downloaded
scene014 t2           2026-10-26           0.0  downloaded
scene01

In [ ]:
#visualisation of the downloaded tiles
import numpy as np
import rasterio
from rasterio.windows import from_bounds
from rasterio.transform import from_origin
from rasterio.enums import ColorInterp

PERIOD = "t2"  # change to "t1" for pre-flood imagery
NODATA = -9999.0
SCENES = [f"scene{i:03d}" for i in range(1, 22)]
scene_root = RAW_DATA_DIR / 's2_all' / PERIOD
out_path = RAW_DATA_DIR / 's2_all' / f"scene001-021_{PERIOD}_rgb.tif"


def stretch_rgb(rgb):
    """Apply a percentile stretch to an RGB image for better visual contrast.

    Raw satellite data often spans a wide dynamic range. A 2-98 percentile stretch clips
    extreme values and maps the middle 96% to the display range, improving visibility of
    detail without losing important information to saturation.

    Args:
        rgb: Array of shape (3, height, width) or (height, width, 3).

    Returns:
        Stretched array normalized to [0, 1].
    """
    rgb = np.moveaxis(rgb, 0, -1).astype("float32")
    p_low, p_high = np.nanpercentile(rgb, (2, 98))
    if p_high == p_low:
        return np.zeros_like(rgb)
    rgb = (rgb - p_low) / (p_high - p_low)
    return np.clip(rgb, 0, 1)


# Find the combined extent and pixel size
bounds_list, res, crs = [], None, None
for s in SCENES:
    with rasterio.open(scene_root / s / 'B04.tif') as src:
        bounds_list.append(src.bounds)
        crs = src.crs
        if res is None:
            res = src.res

minx = min(b.left for b in bounds_list)
miny = min(b.bottom for b in bounds_list)
maxx = max(b.right for b in bounds_list)
maxy = max(b.top for b in bounds_list)

width = int(round((maxx - minx) / res[0]))
height = int(round((maxy - miny) / res[1]))
master_transform = from_origin(minx, maxy, res[0], res[1])

# Canvas starts as nodata everywhere
rgb_canvas = np.full((3, height, width), NODATA, dtype=np.float32)

def to_display(dn):
    """Convert L2A digital numbers to 0-255 display values (float32)."""
    refl = np.clip((dn.astype(np.float32) - 1000.0) / 10000.0, 0.0, 1.0)
    return refl * 255.0

for s in SCENES:
    scene_dir = scene_root / s
    with rasterio.open(scene_dir / 'B04.tif') as ref:
        bounds = ref.bounds
        h, w = ref.height, ref.width

    window = from_bounds(*bounds, transform=master_transform)
    row_off = int(round(window.row_off))
    col_off = int(round(window.col_off))

    bands = np.stack([
        rasterio.open(scene_dir / f"{band}.tif").read(1)
        for band in ["B04", "B03", "B02"]
    ])  # [3, H, W]

    # A pixel is valid only if all three bands have data (DN 0 = nodata)
    valid = np.all(bands != 0, axis=0)
    display = to_display(bands)
    display = np.where(valid[None, :, :], display, NODATA)

    # Clip the scene's footprint to the canvas
    r0, c0 = max(row_off, 0), max(col_off, 0)
    r1, c1 = min(row_off + h, height), min(col_off + w, width)
    sr0, sc0 = r0 - row_off, c0 - col_off
    sr1, sc1 = sr0 + (r1 - r0), sc0 + (c1 - c0)

    rgb_canvas[:, r0:r1, c0:c1] = display[:, sr0:sr1, sc0:sc1]

# Apply the percentile stretch to valid pixels only
valid_mask = rgb_canvas[0] != NODATA
canvas_nan = np.where(rgb_canvas == NODATA, np.nan, rgb_canvas)

stretched = stretch_rgb(canvas_nan)                 # (H, W, 3), values in [0, 1]
stretched = np.moveaxis(stretched, -1, 0) * 255.0   # back to (3, H, W), 0-255 scale
rgb_canvas = np.where(valid_mask[None, :, :], stretched, NODATA).astype(np.float32)

# Save with nodata = -9999 (black pixels stay 0.0 and remain visible)
profile = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 3,
    "dtype": "float32",
    "crs": crs,
    "transform": master_transform,
    "nodata": NODATA,
    "compress": "lzw",
}
with rasterio.open(out_path, "w", **profile) as dst:
    dst.write(rgb_canvas)
    dst.colorinterp = (ColorInterp.red, ColorInterp.green, ColorInterp.blue)
    dst.descriptions = ("Red", "Green", "Blue")

print(f"Saved {out_path}")
print(f"Size: {width} x {height} pixels at {res[0]} m")


RasterioIOError: c:\Users\herol\Documents\sds320_project\data\processed\s2_sub_test\t2\scene002\B04.tif: No such file or directory

## loading into data loader

In [ ]:
# data set class v2 (delete cell above if it works)
#creates class with additional information for each tile: bounds and transform

from pathlib import Path
import rasterio
import torch
from rasterio.transform import array_bounds
from rasterio.windows import Window
from rasterio.windows import transform as window_transform
from torch.utils.data import Dataset


class ChangeDetectionTiles(Dataset):
    def __init__(self, before_root, after_root, mask_root, tile_size=256, stride=256, bands=("B04", "B03", "B02")):
        self.before_root = Path(before_root)
        self.after_root = Path(after_root)
        self.mask_root = Path(mask_root)
        self.tile_size = tile_size
        self.stride = stride
        self.bands = bands

        self.scenes = sorted([p.name for p in self.before_root.iterdir() if p.is_dir()])
        self.scene_transforms = {}
        self.index = []

        for scene in self.scenes:
            with rasterio.open(self.before_root / scene / f"{bands[0]}.tif") as src:
                h, w = src.height, src.width
                self.scene_transforms[scene] = src.transform

            for row in range(0, h - tile_size + 1, stride):
                for col in range(0, w - tile_size + 1, stride):
                    self.index.append((scene, row, col))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        scene, row, col = self.index[idx]

        before_arrays = []
        after_arrays = []

        for band in self.bands:
            with rasterio.open(self.before_root / scene / f"{band}.tif") as src:
                before_patch = src.read(1, window=((row, row + self.tile_size), (col, col + self.tile_size)))
            with rasterio.open(self.after_root / scene / f"{band}.tif") as src:
                after_patch = src.read(1, window=((row, row + self.tile_size), (col, col + self.tile_size)))

            before_arrays.append(torch.from_numpy(before_patch))
            after_arrays.append(torch.from_numpy(after_patch))

        with rasterio.open(self.mask_root / f"{scene}.tif") as src:
            mask = src.read(1, window=((row, row + self.tile_size), (col, col + self.tile_size)))

        # Georeferencing of this tile: its own affine transform and bounding box
        scene_tf = self.scene_transforms[scene]
        tile_window = Window(col, row, self.tile_size, self.tile_size)
        tile_tf = window_transform(tile_window, scene_tf)
        # array_bounds returns (west, south, east, north)
        tile_bounds = array_bounds(self.tile_size, self.tile_size, tile_tf)

        before = torch.stack(before_arrays, dim=0)
        after = torch.stack(after_arrays, dim=0)

        return {
            "image": torch.stack([before, after], dim=0).float(),  # [2, C, H, W]
            "mask": torch.from_numpy(mask).float().unsqueeze(0),   # [1, H, W]
            "scene": scene,
            "row": row,
            "col": col,
            "transform": tile_tf,
            "bounds": tile_bounds,
        }


def collate_fn(batch):
    return {
        "image": torch.stack([x["image"] for x in batch]),  # [B, 2, C, H, W]
        "mask": torch.stack([x["mask"] for x in batch]),    # [B, 1, H, W]
        "scene": [x["scene"] for x in batch],
        "row": [x["row"] for x in batch],
        "col": [x["col"] for x in batch],
        "transform": [x["transform"] for x in batch],
        "bounds": [x["bounds"] for x in batch],
    }


In [34]:
from torch.utils.data import DataLoader

s2_all = RAW_DATA_DIR / 's2_all'
master_mask_path = s2_all / 'flood_mask_master.tif'
test_mask_dir = s2_all / 'mask'
test_mask_dir.mkdir(parents=True, exist_ok=True)

sample_tile_size = 128
sample_stride = 64
bands = ["B04", "B03", "B02"]  # Red, Green, Blue bands for Sentinel-2

# Create one mask per scene, cut from the master mask on the scene's own grid
with rasterio.open(master_mask_path) as master:
    for scene_dir in sorted((s2_all / 't1').iterdir()):
        if not scene_dir.is_dir():
            continue
        ref_path = scene_dir / 'B04.tif'
        with rasterio.open(ref_path) as ref:
            ref_profile = ref.profile.copy()
            ref_shape = (ref.height, ref.width)
            ref_bounds = ref.bounds

        window = from_bounds(*ref_bounds, transform=master.transform)
        mask = master.read(1, window=window, boundless=True, fill_value=0,
                           out_shape=ref_shape)

        out_profile = ref_profile.copy()
        out_profile.update(count=1, dtype=master.dtypes[0], nodata=0)
        with rasterio.open(test_mask_dir / f"{scene_dir.name}.tif", 'w', **out_profile) as dst:
            dst.write(mask, 1)

# Build the test dataset from the new downloads
test_ds = ChangeDetectionTiles(
    s2_all / 't1',
    s2_all / 't2',
    test_mask_dir,
    tile_size=sample_tile_size,
    stride=sample_stride,
    bands=bands,
)

test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, collate_fn=collate_fn)
print(f"Number of test samples: {len(test_ds)}")


Number of test samples: 2814


In [35]:
from pathlib import Path

filename = "sampletiles_128_stride64_fcsiamconc_resnet50_20ep"
RESULTS_DIR = PROJECT_DIR / "results"

ckpt_dir = RESULTS_DIR / "logs" / filename
ckpts = sorted(ckpt_dir.rglob("*.ckpt"), key=lambda p: p.stat().st_mtime)
for c in ckpts:
    print(c)


c:\Users\herol\Documents\sds320_project\results\logs\sampletiles_128_stride64_fcsiamconc_resnet50_20ep\version_0\checkpoints\epoch=19-step=160.ckpt


In [36]:
from torchgeo.trainers import ChangeDetection

ckpt_path = ckpts[-1]  # or the checkpoint you chose
task = ChangeDetection.load_from_checkpoint(ckpt_path, map_location="cpu")
#task.eval()


In [17]:
from lightning.pytorch import Trainer
trainer = Trainer(logger=False, enable_checkpointing=False)

trainer.test(model = task, dataloaders = test_loader, ckpt_path=ckpt_path)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Restoring states from the checkpoint path at c:\Users\herol\Documents\sds320_project\results\logs\sampletiles_128_stride64_fcsiamconc_resnet50_20ep\version_0\checkpoints\epoch=19-step=160.ckpt
c:\Users\herol\miniconda3_v2\envs\sds320_v8\Lib\site-packages\lightning\pytorch\trainer\call.py:283: Be aware that when using `ckpt_path`, callbacks used to create the checkpoint need to be provided during `Trainer` instantiation. Please add the following callbacks: ["ModelCheckpoint{'monitor': None, 'mode': 'min', 'every_n_train_steps': 0, 'every_n_epochs': 1, 'train_time_interval': None}"].
Loaded model weights from the checkpoint at c:\Users\herol\Documents\sds320_project\results\logs\sampletil

Output()

c:\Users\herol\miniconda3_v2\envs\sds320_v8\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\herol\miniconda3_v2\envs\sds320_v8\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


c:\Users\herol\miniconda3_v2\envs\sds320_v8\Lib\site-packages\lightning\pytorch\utilities\data.py:79: Trying to 
infer the `batch_size` from an ambiguous collection. The batch size we found is 8. To avoid any miscalculations, 
use `self.log(..., batch_size=batch_size)`.

c:\Users\herol\miniconda3_v2\envs\sds320_v8\Lib\site-packages\lightning\pytorch\utilities\data.py:79: Trying to 
infer the `batch_size` from an ambiguous collection. The batch size we found is 2. To avoid any miscalculations, 
use `self.log(..., batch_size=batch_size)`.

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   test_AverageAccuracy    │    0.9826580286026001     │
│    test_AverageF1Score    │     0.612650454044342     │
│ test_AverageJaccardIndex  │    0.44159775972366333    │
│   test_AveragePrecision   │    0.6205806136131287     │
│    test_AverageRecall     │    0.6049203872680664     │
│   test_OverallAccuracy    │    0.9826580286026001     │
│    test_OverallF1Score    │     0.612650454044342     │
│ test_OverallJaccardIndex  │    0.44159775972366333    │
│   test_OverallPrecision   │    0.6205806136131287     │
│    test_OverallRecall     │    0.6049203872680664     │
│         test_loss         │    0.35756969451904297    │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.35756969451904297,
  'test_AverageAccuracy': 0.9826580286026001,
  'test_AverageF1Score': 0.612650454044342,
  'test_AverageJaccardIndex': 0.44159775972366333,
  'test_AveragePrecision': 0.6205806136131287,
  'test_AverageRecall': 0.6049203872680664,
  'test_OverallAccuracy': 0.9826580286026001,
  'test_OverallF1Score': 0.612650454044342,
  'test_OverallJaccardIndex': 0.44159775972366333,
  'test_OverallPrecision': 0.6205806136131287,
  'test_OverallRecall': 0.6049203872680664}]

In [37]:
# Predictions
preds = trainer.predict(model=task, dataloaders=test_loader, ckpt_path=ckpt_path)

Restoring states from the checkpoint path at c:\Users\herol\Documents\sds320_project\results\logs\sampletiles_128_stride64_fcsiamconc_resnet50_20ep\version_0\checkpoints\epoch=19-step=160.ckpt
c:\Users\herol\miniconda3_v2\envs\sds320_v8\Lib\site-packages\lightning\pytorch\trainer\call.py:283: Be aware that when using `ckpt_path`, callbacks used to create the checkpoint need to be provided during `Trainer` instantiation. Please add the following callbacks: ["ModelCheckpoint{'monitor': None, 'mode': 'min', 'every_n_train_steps': 0, 'every_n_epochs': 1, 'train_time_interval': None}"].
Loaded model weights from the checkpoint at c:\Users\herol\Documents\sds320_project\results\logs\sampletiles_128_stride64_fcsiamconc_resnet50_20ep\version_0\checkpoints\epoch=19-step=160.ckpt


Output()

c:\Users\herol\miniconda3_v2\envs\sds320_v8\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\herol\miniconda3_v2\envs\sds320_v8\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


In [ ]:
#save prediction output as .tif
import numpy as np
import rasterio
import torch

NODATA = -9999.0
APPLY_SIGMOID = True

with rasterio.open(S2_ROOT / "flood_mask_master.tif") as ref:
    H, W = ref.height, ref.width
    master_tf = ref.transform
    crs = ref.crs

pix = master_tf.a  # 10 m
value_sum = np.zeros((H, W), dtype=np.float32)
count = np.zeros((H, W), dtype=np.uint16)

for batch, preds_batch in zip(test_loader, preds):
    out = torch.sigmoid(preds_batch) if APPLY_SIGMOID else preds_batch
    out = out.detach().cpu().numpy()  # [B, 1, h, w]

    for i in range(out.shape[0]):
        tile_tf = batch["transform"][i]
        # Offset of this tile's top-left corner on the master grid
        col0 = int(round((tile_tf.c - master_tf.c) / pix))
        row0 = int(round((master_tf.f - tile_tf.f) / pix))

        patch = out[i, 0]
        h, w = patch.shape

        r0, c0 = max(row0, 0), max(col0, 0)
        r1, c1 = min(row0 + h, H), min(col0 + w, W)
        if r1 <= r0 or c1 <= c0:
            continue

        value_sum[r0:r1, c0:c1] += patch[r0 - row0 : r1 - row0, c0 - col0 : c1 - col0]
        count[r0:r1, c0:c1] += 1

canvas = np.full((H, W), NODATA, dtype=np.float32)
covered = count > 0
canvas[covered] = value_sum[covered] / count[covered]
print(f"Covered pixels: {int(covered.sum())} of {H * W}")

profile = {
    "driver": "GTiff", "height": H, "width": W, "count": 1,
    "dtype": "float32", "crs": crs, "transform": master_tf,
    "nodata": NODATA, "compress": "lzw",
}

out_path = RESULTS_DIR / "predictions" / f"pred_{filename}_all.tif"
out_path.parent.mkdir(parents=True, exist_ok=True)
with rasterio.open(out_path, "w", **profile) as dst:
    dst.write(canvas, 1)
print(f"Saved: {out_path}")


Covered pixels: 13516800 of 45990008
Saved: c:\Users\herol\Documents\sds320_project\results\predictions\pred_sampletiles_128_stride64_fcsiamconc_resnet50_20ep_all.tif
